In [1]:
import os
import sys
import torch
import torch.nn as nn
from torch.nn import functional as F

# Configuration des chemins vers le backend
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
BACKEND_PATH = os.path.join(PROJECT_ROOT, "backend")

if BACKEND_PATH not in sys.path:
    sys.path.insert(0, BACKEND_PATH)

# Importation directe de votre architecture partagée
from app.models.nanogpt import NanoGPT

# Choix du périphérique (GPU si disponible pour accélérer, sinon CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Environnement prêt ✓ | Utilisation du périphérique : {device.upper()}")

Environnement prêt ✓ | Utilisation du périphérique : CPU


In [2]:
# 1. Chargement du dataset français nettoyé
dataset_path = os.path.join(PROJECT_ROOT, "data/fr", "marque_tech.txt")
with open(dataset_path, 'r', encoding='utf-8') as f:
    text = f.read()

# 2. Extraction du dictionnaire de caractères (Vocabulaire de base)
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Taille du vocabulaire de caractères : {vocab_size}")

# Tables de correspondance (Mapping)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]          # Texte -> Liste d'entiers
decode = lambda l: ''.join([itos[i] for i in l])  # Liste d'entiers -> Texte

# 3. Séparation en train/val (90% entraînement, 10% validation)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Données d'entraînement : {len(train_data)} tokens | Validation : {len(val_data)} tokens ✓")

Taille du vocabulaire de caractères : 25
Données d'entraînement : 1224 tokens | Validation : 136 tokens ✓


In [3]:
# Hyperparamètres de forme pour le flux de tenseurs
batch_size = 32  # Nombre de séquences traitées en parallèle
block_size = 16  # Longueur maximale du contexte (historique de caractères)

def get_batch(split):
    data_split = train_data if split == 'train' else val_data
    # Sélection d'indices de départ aléatoires
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix]) # Cible = décalée de 1 caractère
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print(f"Forme du batch d'entrée X : {xb.shape} (Batch, Contexte)")
print(f"Forme du batch cible Y    : {yb.shape}")

Forme du batch d'entrée X : torch.Size([32, 16]) (Batch, Contexte)
Forme du batch cible Y    : torch.Size([32, 16])


In [4]:
# Configuration des hyperparamètres du Transformer
model_base = NanoGPT(
    vocab_size=vocab_size, 
    n_embed=64,    # Dimension de l'espace de plongement (embedding)
    n_head=4,      # Nombre de têtes d'auto-attention
    n_layer=4,     # Nombre de blocs de couches de Transformers
    block_size=block_size
)
model_base = model_base.to(device)

optimizer = torch.optim.AdamW(model_base.parameters(), lr=1e-3)
max_iters = 1000  # Nombre d'itérations pour valider la convergence de base
eval_interval = 200

print("Début de la validation de l'apprentissage (Training Loop)...")

for iteration in range(max_iters):
    # Évaluation régulière sur les ensembles train et val
    if iteration % eval_interval == 0 or iteration == max_iters - 1:
        model_base.eval()
        with torch.no_grad():
            x_t, y_t = get_batch('train')
            x_v, y_v = get_batch('val')
            _, loss_train = model_base(x_t, y_t)
            _, loss_val = model_base(x_v, y_v)
        print(f"Itération {iteration:4d} | Train Loss: {loss_train.item():.4f} | Val Loss: {loss_val.item():.4f}")
        model_base.train()

    # Phase d'apprentissage classique
    xb, yb = get_batch('train')
    logits, loss = model_base(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("\nEntraînement de validation complété ! L'architecture est stable et compile sans erreurs. ✓")

Début de la validation de l'apprentissage (Training Loop)...
Itération    0 | Train Loss: 3.2702 | Val Loss: 3.2368
Itération  200 | Train Loss: 1.5104 | Val Loss: 2.7016
Itération  400 | Train Loss: 0.8793 | Val Loss: 2.8720
Itération  600 | Train Loss: 0.5946 | Val Loss: 3.2561
Itération  800 | Train Loss: 0.5064 | Val Loss: 3.3590
Itération  999 | Train Loss: 0.4362 | Val Loss: 3.6660

Entraînement de validation complété ! L'architecture est stable et compile sans erreurs. ✓


In [5]:
model_base.eval()

# On initialise le contexte avec un token de départ (ex: index 0)
context = torch.zeros((1, 1), dtype=torch.long, device=device)

# Demander au modèle de générer 100 caractères consécutifs
print("--- Test d'invention de noms bruts (nanoGPT Non-Conditionnel) ---")
generated_tokens = model_base.generate(context, max_new_tokens=100)[0].tolist()

# Décodage des nombres en caractères lisibles
raw_output = decode(generated_tokens)
print(raw_output)
print("\nSprint 2 validé ! Le modèle sait prédire et articuler des structures de caractères. ✓")

--- Test d'invention de noms bruts (nanoGPT Non-Conditionnel) ---

bynova

Sprint 2 validé ! Le modèle sait prédire et articuler des structures de caractères. ✓
